# QC0426 — Measure

`metadata.hera.policy` — scientific record: append new cells; never edit or
delete executed ones. (Who *executes* a measure cell is set by the session
mode — see the brief.)

Confirm each `# <-- verify` (rig: addresses) and `# <-- validate` (device:
operating limits, wiring) line at bring-up. Cells marked **PATTERN** /
**PROPOSAL** carry `<PLACEHOLDER>` values — review before running.

## Setup

In [1]:
import os
import time

import numpy as np
import scipy.constants as cst

import qcodes
from qcodes import validators
from qcodes.instrument_drivers.stanford_research import SR830
from qcodes.instrument_drivers.tektronix.Keithley_2400 import Keithley_2400

import sweep

from hera.analysis import GateCalibration, field_invB

from drivers.FastBi1K5 import FastBi1K5
from drivers.Lakeshore_335 import Lakeshore_335
from drivers.SRS_DC205 import SRS_DC205
from drivers.async_station import async_station

In [2]:
main_dir = os.getcwd()
sweep.set_basedir(os.path.join(main_dir, "data"))
main_dir

'C:\\Users\\DGG\\VTI_local_data\\zack\\20260621-QC0426'

# Connect Instruments

## Magnet

In [3]:
magnet = FastBi1K5('magnet', '192.168.0.10')  # <-- verify address
magnet.status()

Connected to: Fast-Bi-1K5 Magnet Controller
---------- Magnet Status ----------
Field:               : 0.003 Tesla
Current:             : 0.020 A
Voltage:             : -0.000 V
Output Enabled       : True
Persistent Switch On : True
Loop Mode            : CC
Update Mode          : NORMAL
-----------------------------------


## Lakeshore

In [4]:
heater = Lakeshore_335('l335_3', 'GPIB0::10::INSTR')  # <-- verify address
print(heater.sample_temperature())
print(heater.vaporizer_temperature())

Connected to: LSCI 335 (serial:LSA2FM7/#######, firmware:1.4) in 0.44s
1.159
0.0


## Lock-ins

In [5]:
sr0 = SR830('sr0', "GPIB0::17::INSTR")  # <-- verify — current channel (Ithaco preamp output)
sr1 = SR830('sr1', "GPIB0::14::INSTR")  # <-- verify — also the AC drive source
sr2 = SR830('sr2', "GPIB0::19::INSTR")  # <-- verify
sr3 = SR830('sr3', "GPIB0::18::INSTR")  # <-- verify

Connected to: Stanford_Research_Systems SR830 (serial:s/n45620, firmware:ver1.07) in 0.09s
Connected to: Stanford_Research_Systems SR830 (serial:s/n83412, firmware:ver1.07) in 0.09s


C:\Users\DGG\measurement_env\.venv\Lib\site-packages\qcodes\instrument\instrument_base.py:188: QCoDeSDeprecationWarning: Parameter ch1_databuffer on instrument sr0 does not correctly pass kwargs to its baseclass. A Parameter class must take `**kwargs` and forward them to its baseclass.
  warnings.warn(
C:\Users\DGG\measurement_env\.venv\Lib\site-packages\qcodes\instrument\instrument_base.py:188: QCoDeSDeprecationWarning: Parameter ch2_databuffer on instrument sr0 does not correctly pass kwargs to its baseclass. A Parameter class must take `**kwargs` and forward them to its baseclass.
  warnings.warn(
C:\Users\DGG\measurement_env\.venv\Lib\site-packages\qcodes\instrument\instrument_base.py:188: QCoDeSDeprecationWarning: Parameter ch1_databuffer on instrument sr1 does not correctly pass kwargs to its baseclass. A Parameter class must take `**kwargs` and forward them to its baseclass.
  warnings.warn(
C:\Users\DGG\measurement_env\.venv\Lib\site-packages\qcodes\instrument\instrument_base.p

Connected to: Stanford_Research_Systems SR830 (serial:s/n41639, firmware:ver1.07) in 0.09s
Connected to: Stanford_Research_Systems SR830 (serial:s/n93552, firmware:ver1.07) in 0.09s


C:\Users\DGG\measurement_env\.venv\Lib\site-packages\qcodes\instrument\instrument_base.py:188: QCoDeSDeprecationWarning: Parameter ch1_databuffer on instrument sr3 does not correctly pass kwargs to its baseclass. A Parameter class must take `**kwargs` and forward them to its baseclass.
  warnings.warn(
C:\Users\DGG\measurement_env\.venv\Lib\site-packages\qcodes\instrument\instrument_base.py:188: QCoDeSDeprecationWarning: Parameter ch2_databuffer on instrument sr3 does not correctly pass kwargs to its baseclass. A Parameter class must take `**kwargs` and forward them to its baseclass.
  warnings.warn(


## Keithleys

In [6]:
k_2400 = Keithley_2400('k_2400', 'GPIB0::24::INSTR')  # <-- verify address — back-gate Keithley
k_2400.mode.set('VOLT')
k_2400.volt.set(0)
k_2400.rangei.set(1e-6)
k_2400.rangev.set(21)
k_2400.volt.step = 0.005
k_2400.nplci.set(1)

Connected to: KEITHLEY INSTRUMENTS INC. 2400 (serial:0734554, firmware:C17   Jul 16 1999 15:20:40/A02  /G/F) in 0.03s


In [7]:
k_2410 = Keithley_2400('k_2410', 'GPIB0::25::INSTR')  # <-- verify address — Si-gate Keithley
k_2410.mode.set('VOLT')
k_2410.volt.set(0)
k_2410.rangei.set(1e-6)
k_2410.rangev.set(21)
k_2410.volt.step = 0.005
k_2410.volt.inter_delay = 0.01  # Si gate dV/dt pacing - dielectric safety
k_2410.nplci.set(1)

Connected to: KEITHLEY INSTRUMENTS INC. 2410 (serial:1293333, firmware:C30   Mar 17 2006 09:29:29/A02  /J/J) in 0.03s


## DC205 (top gate)

In [8]:
dc205 = SRS_DC205("dc205", "ASRL7::INSTR")  # <-- verify address — top-gate DC205
dc205.ramp_rate = 2  # V/s

Connected to: Stanford_Research_Systems DC205 (serial:s/n20500514, firmware:ver1.80) in 0.03s


In [9]:
# slot: gate-bindings — role aliases (k_tg/k_bg/k_si) from the device card's
# Gate stack section (which instrument drives which gate). The dc205_volt
# wrapper is generic; the k_* role map is device-specific. Each value
# `# <-- validate`.

# Sweepable top-gate parameter — appears in the data as column 'DC_bias'.
def dcbias(vg):
    dc205.ramp_voltage(vg)

dc205_volt = qcodes.Parameter('DC_bias', unit='V', set_cmd=dcbias)

# Gate role aliases (recipes use k_tg / k_bg / k_si):
k_tg = dc205_volt   # top gate <- DC205 (fast axis is dc205_volt, NOT k_tg.volt)   # <-- validate
k_bg = k_2400       # back gate <- Keithley 2400                                     # <-- validate
k_si = k_2410       # Si gate   <- Keithley 2410                                     # <-- validate

# SAFETIES

Run immediately after instrument bring-up, before any sweep.

In [10]:
# slot: gate-validators — operating limits from the device card's gate-
# operating-limits table; commit EXACTLY the card numbers. Each value
# `# <-- validate`. Instrument-level fences go on the RAW instruments
# (dc205 / k_2400 / k_2410); direct calls bypass the role-alias wrapper.
TG_MIN, TG_MAX = -6.0, +7.0    # top gate (dc205_volt), V    # <-- validate
BG_MIN, BG_MAX = -4.5, +4.5    # back gate (k_2400), V        # <-- validate
SI_MIN, SI_MAX = -50.0, +50.0  # Si gate (k_2410), V          # <-- validate
dc205_volt.vals = validators.Numbers(TG_MIN, TG_MAX)
dc205.voltage.vals = validators.Numbers(TG_MIN, TG_MAX)  # instrument-level: direct dc205 calls bypass dc205_volt
k_2400.volt.vals = validators.Numbers(BG_MIN, BG_MAX)
k_2410.volt.vals = validators.Numbers(SI_MIN, SI_MAX)

# Limit lists for sanitize_tg_bg()
tg_lims, bg_lims = [TG_MIN, TG_MAX], [BG_MIN, BG_MAX]

# Session field cap — machine-enforced: magnet.field is a qcodes Parameter,
# so this validator rejects any magnet.field(target) with |B| > B_SESSION_CAP
# before any hardware traffic (and the non-blocking start_field_ramp re-checks
# it too). (Driver max_field_T = 14.0 remains as backstop.)
B_SESSION_CAP = 8.0  # Tesla    # <-- validate (session cap from card/brief)
magnet.field.vals = validators.Numbers(-B_SESSION_CAP, B_SESSION_CAP)

# Temperature setpoint cap — TYPO GUARD ONLY (sample_setpoint(3000) vs 300).
# The real safe T/B envelope is a rig-card item.
T_SESSION_CAP = 310.0  # K    # <-- verify
heater.sample_setpoint.vals = validators.Numbers(0, T_SESSION_CAP)
heater.vaporizer_setpoint.vals = validators.Numbers(0, T_SESSION_CAP)

# Helpers

In [11]:
def spoints(a, b, c):
    return np.linspace(a, b, c)


def sanitize_tg_bg(vtg_vec, vbg_vec, tg_lims, bg_lims):
    """
    Given a list of tg and bg voltages, removes any points which are outside tg_lims and bg_lims
    """
    vtg_new_vec = []
    vbg_new_vec = []
    for ind in range(len(vtg_vec)):
        vtg, vbg = vtg_vec[ind], vbg_vec[ind]
        if vtg < tg_lims[0] or vtg > tg_lims[1] or vbg < bg_lims[0] or vbg > bg_lims[1]:
            pass
        else:
            vtg_new_vec.append(vtg)
            vbg_new_vec.append(vbg)
    return np.array(vtg_new_vec), np.array(vbg_new_vec)

In [12]:
# slot: geometry — gate-stack thicknesses from the device card's Gate-stack
# geometry (geometry prior — calibrate cnp-and-dline-calibration before
# trusting derived scales). Each value `# <-- validate`.
t_tg = 15.6e-9   # top-gate hBN thickness, m    # <-- validate
t_bg = 12e-9     # back-gate hBN thickness, m    # <-- validate
t_bgSiO2 = 0     # SiO2 term, m — zero for a metal/graphite back gate    # <-- validate

# Session gate calibration. cal.n_D(V_tg, V_bg) -> (n [m^-2], D [V/m]);
# set n_offset / D_offset from cnp-and-dline-calibration once live.
cal = GateCalibration.from_thicknesses(t_tg, t_bg, t_bgSiO2)

# Measurement config

In [13]:
# slot: meas-config — sr0_gain + channel/contact assignments from the device
# card's Measurement setup (current channel + gain, provided contact
# configuration). Each value `# <-- validate`. Always set 'comments' per run —
# structured, current, no inherited text.
meas_config = {
    "sr0": "drain current on B7",  # <-- validate
    "sr0_gain": -1e6,  # ithaco gain    # <-- validate
    "sr1": "T3-T4 (Rxx)",  # <-- validate
    "sr2": "B4-B5 (Rxx)",  # <-- validate
    "sr3": "T5-B6 (Rxy, Hall)",  # <-- validate
    "bias_resistor": 100e6,
    "bias_voltage": sr1.amplitude(),
    "lockin_freq": sr1.frequency(),
    "sr0_sens": sr0.sensitivity(),
    "sr1_sens": sr1.sensitivity(),
    "sr2_sens": sr2.sensitivity(),
    "sr3_sens": sr3.sensitivity(),
    "sample_temperature:": heater.sample_temperature(),  # key verbatim (trailing ':') to match this cooldown's metadata
    "Si (V)": k_2410.volt(),
    "comments": "<SET PER RUN>",
}


## QSwitch (contact mux) — LIVE

Contact-mux bring-up and the line/breakout maps are device-specific — from the
device card's Contact map section. `qsw.reset()` is **always forbidden** — it
opens the Si standing closures and drops the Si gate.

In [14]:
from drivers.QSwitch import QSwitch
from drivers.contact_mux import ContactMux

qsw = QSwitch('qswitch', 'ASRL4::INSTR')  # <-- verify address — QSwitch

# Contact label -> QSwitch line. Si standing-closure lines are EXCLUDED.
LINES = {  # <-- validate (from card contact map)
    "T1": 1,  "T2": 3,  "T3": 4,  "T4": 6,  "T5": 7,  "T6": 12, "T7": 13,
    "B1": 8,  "B2": 17, "B3": 23, "B4": 22, "B5": 20, "B6": 19, "B7": 18, "B8": 15,
}
# Instrument port -> front-panel BNC tap.
BREAKOUTS = {  # <-- validate (from card instrument taps)
    "source": 7, "drain": 8,
    "sr1+": 1, "sr1-": 2, "sr2+": 3, "sr2-": 4, "sr3+": 5, "sr3-": 6,
}
SETTLE_S = 4  # <-- validate (calibrated relay settle)

mux = ContactMux(qsw, lines=LINES, breakouts=BREAKOUTS, settle_s=SETTLE_S)
# qsw.reset() ALWAYS FORBIDDEN — drops the Si gate (standing closures).

Connected to: Quantum Machines QSwitch (serial:37, firmware:2.0) in 0.06s


# HERA — session start (2026-06-21)

## HERA — session start: kernel-binding + VERIFY probe (read-only, pre-grant)

Gates **A** (instruments alive) and **B** (SAFETIES committed = device-card limits; B cap **8.0 T** on the `magnet.field` validator) are green from the bring-up outputs. This **read-only** cell closes the remaining VERIFY reads and doubles as the first-E1 **kernel-binding probe** — it confirms my execute path is bound to your instrument-carrying kernel. It moves no hardware (~5 s): magnet field/state, sample+vaporizer T, lock-in τ / filter-slope / sensitivity (rig card: read & record at bring-up — the kit does not set these), standing gate voltages, and a readback of the committed validators.

In E0 you may run it now to close VERIFY; under E1 it is my first execute. **Awaiting "go for autonomy" before I execute anything.** Next after this: Gate C contact liveness on the provided config, then the 0 T TG–BG gate map (Step 1 → `cnp-and-dline-calibration`).


In [15]:
# Read-only: kernel-binding probe + VERIFY-completion. No hardware moves.
print("== MAGNET ==")
print("field (meas):", magnet.field(), "T;  status field:", magnet._status()['field'], "T")
print("PS on:", magnet.persistent_switch_on(), " output enabled:", magnet.output_enabled())

print("== THERMOMETRY ==")
print("sample T:", heater.sample_temperature(), "K;  vaporizer T:", heater.vaporizer_temperature(), "K")

print("== LOCK-INS (tc / slope / sens) ==")
for name, sr in [("sr0", sr0), ("sr1", sr1), ("sr2", sr2), ("sr3", sr3)]:
    print(name, " tc:", sr.time_constant(), "s  slope:", sr.filter_slope(), "  sens:", sr.sensitivity())
print("sr1 drive amplitude:", sr1.amplitude(), "V  freq:", sr1.frequency(), "Hz")

print("== GATES (standing values) ==")
print("dc205 top gate:", dc205.voltage(), "V")
print("k_2400 back gate:", k_2400.volt(), "V")
print("k_2410 Si gate:", k_2410.volt(), "V")

print("== COMMITTED VALIDATORS ==")
print("magnet.field.vals:", magnet.field.vals)
print("dc205_volt.vals:", dc205_volt.vals, " dc205.voltage.vals:", dc205.voltage.vals)
print("k_2400.volt.vals:", k_2400.volt.vals)
print("k_2410.volt.vals:", k_2410.volt.vals)

print("== NAMESPACE / SWEEP STACK ==")
print("async_station:", async_station, " mux:", mux, " cal:", cal)
print("KERNEL BINDING OK")


== MAGNET ==
field (meas): 0.0026835753424657533 T;  status field: 0.0026857671232876712 T
PS on: True  output enabled: True
== THERMOMETRY ==
sample T: 1.159 K;  vaporizer T: 0.0 K
== LOCK-INS (tc / slope / sens) ==
sr0  tc: 0.3 s  slope: 24   sens: 0.05
sr1  tc: 0.3 s  slope: 24   sens: 0.0001
sr2  tc: 0.3 s  slope: 24   sens: 0.0001
sr3  tc: 0.3 s  slope: 24   sens: 0.0002
sr1 drive amplitude: 2.0 V  freq: 13.0 Hz
== GATES (standing values) ==
dc205 top gate: 0.0 V


k_2400 back gate: 0.0 V
k_2410 Si gate: 0.0 V
== COMMITTED VALIDATORS ==
magnet.field.vals: <Numbers -8.0<=v<=8.0>
dc205_volt.vals: <Numbers -6.0<=v<=7.0>  dc205.voltage.vals: <Numbers -6.0<=v<=7.0>
k_2400.volt.vals: <Numbers -4.5<=v<=4.5>
k_2410.volt.vals: <Numbers -50.0<=v<=50.0>
== NAMESPACE / SWEEP STACK ==
async_station: <class 'drivers.async_station.async_station'>  mux: <drivers.contact_mux.ContactMux object at 0x00000118D0A71880>  cal: GateCalibration(c_tg=0.0017027284266923077, c_bg=0.0022135469547, n_offset=0.0, D_offset=0.0)
KERNEL BINDING OK


## HERA — Gate C: contact liveness on the provided configuration (E1, first-session branch)

First-session Gate C: route the **provided** contact config (source T2, drain B7, Rxx T3–T4 / B4–B5, Rxy T5–B6) and take a single-point read of every channel at the standing condition (gates 0,0,0; field ~0). Purpose: confirm the contacts are alive, give sane four-terminal signals, and that the **current** lock-in config (2.0 V / ~20 nA drive, sens 1e-4) does not overload — this read sets the bias/sensitivity for the upcoming 0 T gate map. No gate moves, no field move; relays move once (settle 4 s). ~10 s. Auto bucket.


In [16]:
# Gate C - contact liveness on the provided configuration (first-session branch).
# Preconditions (cell assumes, does not set):
T = heater.sample_temperature(); B = magnet.field()
print(f"PRECOND: sample T = {T:.3f} K (expect ~1.16);  field = {B:.4f} T (expect ~0)")
print(f"PRECOND: gates  TG={dc205.voltage():.3f}  BG={k_2400.volt():.3f}  Si={k_2410.volt():.3f} V (expect 0,0,0)")
assert T < 2.0, "sample T unexpectedly high"
assert abs(B) < 0.05, "field not near zero"

# Route the PROVIDED contact configuration (relays move; settle 4 s).
mux.select({"source": "T2", "drain": "B7",
            "sr1+": "T3", "sr1-": "T4",
            "sr2+": "B4", "sr2-": "B5",
            "sr3+": "T5", "sr3-": "B6"})

sr0_gain = -1e6
Idrain = sr0.X() / sr0_gain          # transimpedance: V/gain -> A
Iexp = sr1.amplitude() / 100e6
print(f"\nDrain current I = {Idrain*1e9:8.3f} nA   (bias {sr1.amplitude()} V / 100 MOhm -> {Iexp*1e9:.1f} nA expected)")

for name, sr in [("sr1 T3-T4 Rxx", sr1), ("sr2 B4-B5 Rxx", sr2), ("sr3 T5-B6 Rxy", sr3)]:
    X = sr.X(); sens = sr.sensitivity()
    R = X / Idrain if Idrain != 0 else float('nan')
    ovl = abs(X) > 0.9*sens
    print(f"{name:14s}: X = {X*1e6:9.2f} uV   R = {R:10.1f} Ohm   sens={sens:.0e}   {'** NEAR/OVERLOAD **' if ovl else 'ok'}")
print(f"\nsr0 drain: X = {sr0.X():.4e} V   sens = {sr0.sensitivity():.0e}")


PRECOND: sample T = 1.159 K (expect ~1.16);  field = 0.0027 T (expect ~0)
PRECOND: gates  TG=0.000  BG=0.000  Si=0.000 V (expect 0,0,0)



Drain current I =   19.943 nA   (bias 2.0 V / 100 MOhm -> 20.0 nA expected)
sr1 T3-T4 Rxx : X =     28.11 uV   R =     1409.5 Ohm   sens=1e-04   ok
sr2 B4-B5 Rxx : X =     24.68 uV   R =     1237.7 Ohm   sens=1e-04   ok
sr3 T5-B6 Rxy : X =      5.30 uV   R =      265.6 Ohm   sens=2e-04   ok

sr0 drain: X = -1.9941e-02 V   sens = 5e-02


## HERA — set CNP/gate-map lock-in regime + inspect gate ramping (E1, Auto)

Move to the device-card **CNP/gate-map regime**: drive 0.5 V (→5 nA through 100 MΩ), and set the four-terminal voltage channels (sr1/sr2/sr3) to **sens 1e-3** — generous headroom so an unknown CNP Rxx (blinded first session) cannot overload (1 mV FS ≈ 200 kΩ at 5 nA; recorded X is the true value regardless, sens only guards overload/noise). Also read out how the gates ramp (step / inter_delay / dc205 ramp_rate) before I move them for the map. No gate moves here (gates stay 0,0,0); reversible. Verifies the new 5 nA operating point and overload status.


In [17]:
import time
# 1. Inspect gate ramp configuration (how gates move when set) - read only.
print("dc205 ramp_rate:", getattr(dc205, 'ramp_rate', '?'), "V/s")
for nm, p in [("k_2400.volt (BG)", k_2400.volt), ("k_2410.volt (Si)", k_2410.volt)]:
    print(f"{nm:18s} step: {p.step}  inter_delay: {p.inter_delay}")

# 2. Set CNP-regime bias + generous sensitivity.
sr1.amplitude(0.5)                      # 0.5 V / 100 MOhm -> 5 nA (device-card CNP regime)
for sr in (sr1, sr2, sr3):
    sr.sensitivity(1e-3)                # 1 mV FS -> headroom to ~200 kOhm at 5 nA
time.sleep(3)                           # settle (tc 0.3 s)

# 3. Verify new operating point (gates still 0,0,0).
sr0_gain = -1e6
Idrain = sr0.X()/sr0_gain
print(f"\nNew drive {sr1.amplitude()} V -> I = {Idrain*1e9:.3f} nA (expect ~5)")
for name, sr in [("sr1 T3-T4", sr1), ("sr2 B4-B5", sr2), ("sr3 T5-B6", sr3)]:
    X = sr.X(); ovl = abs(X) > 0.9*sr.sensitivity()
    print(f"{name}: X = {X*1e6:8.2f} uV   R = {X/Idrain:9.1f} Ohm   sens={sr.sensitivity():.0e}   {'** OVLD **' if ovl else 'ok'}")
print("gates:", dc205.voltage(), k_2400.volt(), k_2410.volt())


dc205 ramp_rate: 2 V/s
k_2400.volt (BG)   step: 0.005  inter_delay: 0
k_2410.volt (Si)   step: 0.005  inter_delay: 0.01



New drive 0.5 V -> I = 5.137 nA (expect ~5)
sr1 T3-T4: X =     7.15 uV   R =    1392.5 Ohm   sens=1e-03   ok
sr2 B4-B5: X =     6.32 uV   R =    1230.0 Ohm   sens=1e-03   ok
sr3 T5-B6: X =     1.19 uV   R =     232.1 Ohm   sens=1e-03   ok


gates: 0.0 0.0 0.0


## HERA — Run 1: 0 T TG–BG gate map (CNP locate + (n,D) cal source) [E1, Auto]

First science measurement (`gatemap` → feeds `cnp-and-dline-calibration`). Coarse-but-full-window raster over the safe gate box, **BG slow (k_2400) / TG fast (dc205_volt, col `DC_bias`)** — the orientation the CNP recipe fits (ridge = V_tg at Rxx-max per V_bg row; ratio = −slope). Vsi held at 0. Bias 0.5 V/5 nA, sr1/2/3 sens 1e-3 (no-overload headroom).

- **Grid:** TG ∈ [−5.9, 6.9] V × 61 (0.21 V/pt); BG ∈ [−4.4, 4.4] V × 25 (0.37 V/row). 1525 pts.
- **Settling:** fast_delay 1.0 s ≈ 3.3τ (tc 0.3 s/24 dB). Slight under-settle, acceptable for the broad CNP ridge and the ratio fit; SdH-grade reads later use the fuller floor.
- **Leakage monitor:** k_2400.curr + k_2410.curr followed every point (DC205 has no current readback). Lakeshore deliberately NOT followed per-point (rig quirk: contends with the Grafana logger).
- **Estimate:** ~34 min (incl. ~6.4 s/row TG ramp-back, 2 V/s). **Checkpoint at BG row ~13 (~18 min):** confirm a CNP ridge is present and inside the window, no overload/leakage → continue / adjust range / abandon.
- **Coverage note:** spans nearly the full validator box; if the CNP line exits the TG window on some BG rows those rows clip (excluded from the fit, reported). Cost of full coverage already paid (whole box swept).

Gates parked to 0,0 at the end. Monitored live via `hera-watch`.


In [18]:
import numpy as np, time

# --- Preconditions (cell assumes, does not set) ---
T = heater.sample_temperature(); B = magnet.field()
print(f"PRECOND: T={T:.3f} K (~1.16)  B={B:.4f} T (~0)  gates TG={dc205.voltage()} BG={k_2400.volt()} Si={k_2410.volt()}")
assert T < 2.0 and abs(B) < 0.05, "precondition fail (T/B)"
assert abs(k_2410.volt()) < 0.01, "Si gate not at 0"

# --- Refresh meas_config to the actual operating point ---
meas_config['sr0_gain'] = -1e6
meas_config['bias_voltage'] = sr1.amplitude()
meas_config['lockin_freq'] = sr1.frequency()
meas_config['sr0_sens'] = sr0.sensitivity()
meas_config['sr1_sens'] = sr1.sensitivity()
meas_config['sr2_sens'] = sr2.sensitivity()
meas_config['sr3_sens'] = sr3.sensitivity()
meas_config['sample_temperature:'] = T
meas_config['Si (V)'] = k_2410.volt()
meas_config['comments'] = "0T TG-BG CNP-locate map; Vsi=0; 0.5V/5nA; provided contacts src T2 drn B7, Rxx T3-T4 & B4-B5, Rxy T5-B6"

# --- Sweep vectors ---
tg_vec = np.linspace(-5.9, 6.9, 61)   # fast = dc205_volt (col 'DC_bias'), inside [-6,7]
bg_vec = np.linspace(-4.4, 4.4, 25)   # slow = k_2400.volt, inside [-4.5,4.5]

# --- Build station fresh; follow raw X/P (gains applied downstream via meas_config) ---
s = async_station(meas_config)
s.add_comment('gatemap')
s.add_comment('QC0426 0T TG-BG CNP-locate->cnp-and-dline-calibration; Vsi=0; src T2 drn B7; 0.5V/5nA; sens 1e-3')
(s.fp(sr0.X).fp(sr0.P)
  .fp(sr1.X).fp(sr1.P)
  .fp(sr2.X).fp(sr2.P)
  .fp(sr3.X).fp(sr3.P)
  .fp(k_2400.curr).fp(k_2410.curr))

# --- Ramp to start corner gently (dc205_volt is set-only: set directly, no getter) ---
k_2400.volt(bg_vec[0])      # 5 mV steps
dc205_volt(tg_vec[0])       # 2 V/s ramp
time.sleep(2)

# --- 2D map: BG slow, TG fast ---
s.megasweep(k_2400.volt, bg_vec, dc205_volt, tg_vec, slow_delay=1.0, fast_delay=1.0)

# --- Park gates ---
dc205_volt(0.0); k_2400.volt(0.0)
print("MAP DONE; gates parked to TG=0, BG=0; Si untouched (0).")


PRECOND: T=1.159 K (~1.16)  B=0.0027 T (~0)  gates TG=0.0 BG=0.0 Si=0.0


2026-06-21 04:11:08,386 [INFO] Starting megasweep with ID 0


2026-06-21 04:11:08,387 [INFO] Minimum duration 0h 25m 50s


2026-06-21 04:46:06,376 [INFO] Completed in 0h 34m 57s


2026-06-21 04:46:06,378 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260621-QC0426\data\0\data.tsv.gz


MAP DONE; gates parked to TG=0, BG=0; Si untouched (0).


## HERA — SdH spot-check setup: move to (n=−3e12 cm⁻², D=0), high-|n| metallic regime [E1, Auto]

Stage the hole-side SdH spot check (investigation Step 3). Move gates along the session D=0 line to **n = −3.0e12 cm⁻²** → V_tg=−1.30, V_bg=−0.99 V (both well inside validators). Switch to the device-card **high-|n| metallic drive: 2 V/20 nA**. Gates moved *first* (at the gentle CNP-regime bias) so R is low before the drive steps up — avoids an overload transient. Then match each voltage-channel sensitivity to the measured signal (~8× headroom; recipe step 3 anti-graininess), and verify no overload + leakage. Reversible config; no field move yet.


In [19]:
import numpy as np, time
# --- Preconditions ---
T = heater.sample_temperature(); B = magnet.field()
print(f"PRECOND T={T:.3f}K  B={B:.4f}T  gates TG={dc205.voltage()} BG={k_2400.volt()} Si={k_2410.volt()}")
assert T < 2.0 and abs(B) < 0.05, "precond T/B"
assert abs(k_2410.volt()) < 0.01, "Si gate != 0"

# --- Target (n=-3e12 cm^-2, D=0) from session calibration ---
ratio = 1.3063; c_tg = 1.7027e-3; n_off = -2.391e15; e = 1.602176634e-19
n_t = -3.0e16  # m^-2  (= -3e12 cm^-2)
Vbg = (n_t - n_off) * e / (2 * ratio * c_tg); Vtg = ratio * Vbg
print(f"target n={n_t/1e4/1e12:.2f}e12 cm^-2, D=0 -> Vtg={Vtg:.3f} Vbg={Vbg:.3f}")
assert -6 < Vtg < 7 and -4.5 < Vbg < 4.5, "gate target out of range"

# --- Move gates (gentle ramps) at the current CNP-regime bias, then step the drive up ---
k_2400.volt(Vbg); dc205_volt(Vtg); time.sleep(2)
sr1.amplitude(2.0); time.sleep(3)   # 2 V / 100 MOhm -> 20 nA
Idrain = sr0.X() / -1e6
print(f"\nI = {Idrain*1e9:.2f} nA (expect ~20);  sr0.X={sr0.X()*1e3:.2f} mV sens={sr0.sensitivity():.0e}")

SENS = [2e-9,5e-9,1e-8,2e-8,5e-8,1e-7,2e-7,5e-7,1e-6,2e-6,5e-6,1e-5,2e-5,5e-5,1e-4,2e-4,5e-4,1e-3,2e-3,5e-3,1e-2,2e-2,5e-2,1e-1]
def pick(x, h=8):
    t = max(abs(x)*h, 2e-9)
    return next((s for s in SENS if s >= t), SENS[-1])
print("\nmatch sensitivities (R at 20 nA, current sens):")
for nm, sr in [("sr1", sr1), ("sr2", sr2), ("sr3", sr3)]:
    X = sr.X(); R = X/Idrain; ns = pick(X)
    print(f"  {nm}: X={X*1e6:7.2f}uV  R={R:8.1f}Ohm  cur={sr.sensitivity():.0e} -> set {ns:.0e}")
    sr.sensitivity(ns)
time.sleep(3)
print("\npost-set verify:")
for nm, sr in [("sr1", sr1), ("sr2", sr2), ("sr3", sr3)]:
    X = sr.X(); fs = abs(X)/sr.sensitivity()*100
    print(f"  {nm}: X={X*1e6:7.2f}uV  sens={sr.sensitivity():.0e}  {fs:4.0f}%FS  {'** OVLD **' if fs>90 else 'ok'}")
print(f"leakage: I_bg={k_2400.curr()*1e12:.0f} pA  I_si={k_2410.curr()*1e12:.0f} pA")


PRECOND T=1.158K  B=0.0027T  gates TG=0.0 BG=0.0 Si=0.0
target n=-3.00e12 cm^-2, D=0 -> Vtg=-1.299 Vbg=-0.994



I = 19.82 nA (expect ~20);  sr0.X=-19.83 mV sens=5e-02

match sensitivities (R at 20 nA, current sens):
  sr1: X=   1.43uV  R=    72.2Ohm  cur=1e-03 -> set 2e-05
  sr2: X=   1.25uV  R=    63.2Ohm  cur=1e-03 -> set 2e-05
  sr3: X=   0.00uV  R=     0.0Ohm  cur=1e-03 -> set 2e-09



post-set verify:
  sr1: X=   1.39uV  sens=2e-05     7%FS  ok
  sr2: X=   1.31uV  sens=2e-05     7%FS  ok
  sr3: X=  -0.00uV  sens=2e-09   109%FS  ** OVLD **


leakage: I_bg=31 pA  I_si=286 pA


## HERA — fix lock-in sensitivities for the field sweep (B=0→7.9 T) [E1, Auto]

The B=0 auto-pick mis-set **sr3 (Rxy)** to 2 nV FS (Hall≈0 at zero field) — it would rail as R_xy grows with B. Set sens from the *expected high-field* signals at n=3e12, 20 nA: R_xy(7.9 T)=B/(n·e)≈1650 Ω → ≈33 µV → sens 1e-4 (≈33% FS at max). Bump sr1/sr2 (Rxx) to 1e-5 (≈14% FS now, ~7× MR/SdH headroom — better ADC usage than 2e-5). No gate/field move.


In [20]:
import time
sr1.sensitivity(1e-5)   # Rxx ~72 Ohm -> 1.4 uV now, headroom for MR/SdH
sr2.sensitivity(1e-5)
sr3.sensitivity(1e-4)   # Rxy up to ~33 uV at 7.9 T
time.sleep(2)
Idrain = sr0.X()/-1e6
print(f"I={Idrain*1e9:.2f} nA")
for nm, sr in [("sr1", sr1), ("sr2", sr2), ("sr3", sr3)]:
    X = sr.X(); fs = abs(X)/sr.sensitivity()*100
    print(f"  {nm}: X={X*1e6:7.2f}uV  R={X/Idrain:8.1f}Ohm  sens={sr.sensitivity():.0e}  {fs:4.0f}%FS  {'OVLD' if fs>90 else 'ok'}")
print(f"tau={sr1.time_constant()}s slope={sr1.filter_slope()}  gates TG={dc205.voltage():.3f} BG={k_2400.volt():.3f}")


I=19.98 nA
  sr1: X=   1.52uV  R=    76.1Ohm  sens=1e-05    15%FS  ok
  sr2: X=   1.21uV  R=    60.3Ohm  sens=1e-05    12%FS  ok
  sr3: X=   0.91uV  R=    45.7Ohm  sens=1e-04     1%FS  ok


tau=0.3s slope=24  gates TG=-1.299 BG=-0.995


## HERA — Run 1: hole-side SdH spot check, n=−3e12 cm⁻², D=0 [E1, Auto]

Investigation Step 3 (`sdh-spot-check`). Fixed-(n,D) field sweep, **uniform in 1/B**, B 0.2→7.9 T (start near current ~0 T, end inside the 8 T cap). Read = 1/B FFT of Rxx (post-hoc, from TSV).

- **Sampling:** F_L = (h/4e)·n = 31.0 T at n=3e12. Δ(1/B) = 1/(3·F_L) = 0.01075 /T (tighter than Nyquist 1/(2F_L)) → **N=455** over the 1/B span 4.87 /T. Guards against the documented fast-tone aliasing failure.
- **Settling:** dwell **2.5 s** ≈ 8τ (τ=0.3 s, 24 dB/oct) — the non-negotiable floor for resolving a weak ML partner tone. magnet.settle_s lowered 3→1 s for the monotonic sweep (field readback lag ~41 ms).
- **Leakage monitor:** k_2400.curr + k_2410.curr followed; Lakeshore NOT followed (rig quirk).
- **Estimate ~85–100 min** (field traverse 7.7 T at ~0.137 T/min ≈ 56 min + 455×~3.7 s dwell). **Checkpoint at ~40 min (~B 2–3 T):** Rxx alive & not overloading, current steady, MR sane → continue/abort.
- **Field left at 7.9 T** at the end (not parked) — the electron-side spot check will descend from there, avoiding a 56-min re-traverse (rig card: don't re-traverse). Gates stay at the hole point until then.


In [21]:
import numpy as np, time
# --- Preconditions (assumes, does not set) ---
T = heater.sample_temperature(); B = magnet.field()
print(f"PRECOND T={T:.3f}K B={B:.4f}T  TG={dc205.voltage():.3f} BG={k_2400.volt():.3f} Si={k_2410.volt():.3f}")
assert T < 2.0, "T high"
assert abs(B) < 0.25, "field not near 0 at start"
assert abs(dc205.voltage()-(-1.299)) < 0.02 and abs(k_2400.volt()-(-0.995)) < 0.02, "gates not at n=-3e12,D=0"
assert abs(k_2410.volt()) < 0.01, "Si != 0"
assert sr1.sensitivity() <= 1e-5 and sr3.sensitivity() <= 1e-4, "sens not set"

if hasattr(magnet, 'settle_s'):
    print("magnet.settle_s", magnet.settle_s, "-> 1.0"); magnet.settle_s = 1.0

# meas_config refresh
meas_config['bias_voltage'] = sr1.amplitude()
meas_config['sr0_sens'] = sr0.sensitivity(); meas_config['sr1_sens'] = sr1.sensitivity()
meas_config['sr2_sens'] = sr2.sensitivity(); meas_config['sr3_sens'] = sr3.sensitivity()
meas_config['sample_temperature:'] = T; meas_config['Si (V)'] = k_2410.volt()
meas_config['comments'] = "SdH spot check hole n=-3e12 D=0; 2V/20nA; B 0.2->7.9T uniform-1/B N=455 dwell2.5s; src T2 drn B7"

# field axis uniform in 1/B  (== hera.analysis.field_invB(0.2, 7.9, 455))
B_lo, B_hi, N = 0.2, 7.9, 455
Bvec = 1.0/np.linspace(1.0/B_lo, 1.0/B_hi, N)
FL = 31.0
print(f"N={N}  d(1/B)={(1/B_lo-1/B_hi)/(N-1):.5f} 1/T  vs 1/(3FL)={1/(3*FL):.5f}  Bvec[0]={Bvec[0]:.3f} Bvec[-1]={Bvec[-1]:.3f}")
assert Bvec.max() < 8.0, "field exceeds cap"

s = async_station(meas_config)
s.add_comment('sdh-spot-check')
s.add_comment('QC0426 hole n=-3e12 D=0; 2V/20nA; B 0.2-7.9T uniform-1/B N=455 dwell2.5s; src T2 drn B7')
(s.fp(sr0.X).fp(sr0.P).fp(sr1.X).fp(sr1.P).fp(sr2.X).fp(sr2.P).fp(sr3.X).fp(sr3.P).fp(k_2400.curr).fp(k_2410.curr))
s.sweep(magnet.field, Bvec, delay=2.5)
print("SPOT CHECK DONE; field left at", round(float(magnet.field()), 3), "T (for descending electron spot check)")


PRECOND T=1.158K B=0.0027T  TG=-1.299 BG=-0.995 Si=0.000
magnet.settle_s 3.0 -> 1.0


N=455  d(1/B)=0.01073 1/T  vs 1/(3FL)=0.01075  Bvec[0]=0.200 Bvec[-1]=7.900
2026-06-21 05:03:43,838 [INFO] Starting sweep with ID 1


2026-06-21 05:03:43,839 [INFO] Minimum duration 0h 18m 57s


2026-06-21 06:28:53,359 [INFO] Completed in 1h 25m 9s


2026-06-21 06:28:53,360 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260621-QC0426\data\1\data.tsv.gz


SPOT CHECK DONE; field left at 7.906 T (for descending electron spot check)


## HERA — coarsen Rxx sensitivity for a clean hole re-run [E1, Auto]

Run 1 clipped sr1/sr2 above B≈2.76 T (railed at 546 Ω on the 1e-5 range) — the high-field, high-SNR SdH was lost and the FFT is muddy. Coarsen sr1/sr2 to **5e-5** (50 µV FS ≈ 2500 Ω at 20 nA) to cover the full MR range to 7.9 T without clipping. Read the actual (now-unclipped) Rxx at 7.9 T to confirm headroom; sr3 (Rxy) stays 1e-4. No gate/field move (field held at 7.9 T from Run 1; gates at the hole point). Next cell: descending 7.9→0.2 T re-run (traverse-free).


In [22]:
import time
B = magnet.field()
print(f"B={B:.4f} T  gates TG={dc205.voltage():.3f} BG={k_2400.volt():.3f} Si={k_2410.volt():.3f}")
assert abs(B-7.9) < 0.1, "field not at 7.9 T"
assert abs(dc205.voltage()-(-1.299)) < 0.02 and abs(k_2400.volt()-(-0.995)) < 0.02, "gates moved"
sr1.sensitivity(5e-5); sr2.sensitivity(5e-5)
time.sleep(3)
Idr = sr0.X()/-1e6
print(f"I={Idr*1e9:.2f} nA  (at 7.9 T, hole point)")
for nm, sr in [("sr1", sr1), ("sr2", sr2), ("sr3", sr3)]:
    X = sr.X(); fs = abs(X)/sr.sensitivity()*100
    print(f"  {nm}: X={X*1e6:7.2f}uV  R={X/Idr:8.1f}Ohm  sens={sr.sensitivity():.0e}  {fs:4.0f}%FS  {'** OVLD **' if fs>90 else 'ok'}")


B=7.9027 T  gates TG=-1.299 BG=-0.995 Si=0.000


I=19.64 nA  (at 7.9 T, hole point)
  sr1: X=  37.73uV  R=  1920.5Ohm  sens=5e-05    75%FS  ok
  sr2: X=  32.12uV  R=  1635.0Ohm  sens=5e-05    64%FS  ok
  sr3: X= -32.83uV  R= -1671.5Ohm  sens=1e-04    33%FS  ok


## HERA — Run 2: hole SdH spot check RE-RUN (clean, full field) n=−3e12, D=0 [E1, Auto]

Run 1 clipped Rxx above 2.76 T → muddy FFT. Re-run at **coarse sens 1e-4** (no clip to 7.9 T; Rxx peaks ~1920 Ω = 38% FS). **Descending 7.9→0.2 T** — traverse-free (field already at 7.9 T from Run 1). Full 1/B record (span 4.87 /T) for a clean FFT; the weak ML-partner tone (if ABA) lives at high field where Run 1 clipped. Same uniform-1/B N=455, dwell 2.5 s. Onset/low-field detail cross-checked against Run 1 (finer range there). ~85 min. **Checkpoint ~40 min (~B 2–3 T):** Rxx alive, not clipping, current steady. Field ends at 0.2 T (electron check then moves gates + ascends). Recipe: `sdh-spot-check` (Step 3).


In [23]:
import numpy as np, time
# --- Preconditions ---
B = magnet.field()
print(f"PRECOND B={B:.4f}T  TG={dc205.voltage():.3f} BG={k_2400.volt():.3f} Si={k_2410.volt():.3f}  T={heater.sample_temperature():.3f}K")
assert abs(B-7.9) < 0.1, "field not at 7.9 T start"
assert abs(dc205.voltage()-(-1.299)) < 0.02 and abs(k_2400.volt()-(-0.995)) < 0.02, "gates not at hole point"
assert abs(k_2410.volt()) < 0.01, "Si != 0"

# coarse sens, no clip to 7.9 T
sr1.sensitivity(1e-4); sr2.sensitivity(1e-4); sr3.sensitivity(1e-4)
time.sleep(3)
Idr = sr0.X()/-1e6
print(f"I={Idr*1e9:.2f} nA  at 7.9T:")
for nm, sr in [("sr1", sr1), ("sr2", sr2), ("sr3", sr3)]:
    X = sr.X(); fs = abs(X)/sr.sensitivity()*100
    print(f"  {nm}: R={X/Idr:8.1f}Ohm {fs:4.0f}%FS {'OVLD' if fs>90 else 'ok'}")
    assert fs < 90, f"{nm} overload at start"

# meas_config refresh
for k,v in [('bias_voltage',sr1.amplitude()),('sr1_sens',sr1.sensitivity()),('sr2_sens',sr2.sensitivity()),
            ('sr3_sens',sr3.sensitivity()),('sample_temperature:',heater.sample_temperature()),('Si (V)',k_2410.volt())]:
    meas_config[k]=v
meas_config['comments']="SdH spot check HOLE RE-RUN n=-3e12 D=0; 2V/20nA coarse sens 1e-4; B 7.9->0.2T descending uniform-1/B N=455 dwell2.5s; src T2 drn B7 (Run1 clipped >2.76T)"

# descending field axis uniform in 1/B (== field_invB(7.9, 0.2, 455))
B_hi, B_lo, N = 7.9, 0.2, 455
Bvec = 1.0/np.linspace(1.0/B_hi, 1.0/B_lo, N)
print(f"Bvec[0]={Bvec[0]:.3f} -> Bvec[-1]={Bvec[-1]:.3f}  d(1/B)={(1/B_lo-1/B_hi)/(N-1):.5f} 1/T")
assert Bvec.max() < 8.0 and abs(Bvec[0]-7.9) < 0.01, "field vector bad"

s = async_station(meas_config)
s.add_comment('sdh-spot-check')
s.add_comment('QC0426 hole RE-RUN n=-3e12 D=0; 2V/20nA sens1e-4; B 7.9->0.2T desc uniform-1/B N=455 dwell2.5s; src T2 drn B7')
(s.fp(sr0.X).fp(sr0.P).fp(sr1.X).fp(sr1.P).fp(sr2.X).fp(sr2.P).fp(sr3.X).fp(sr3.P).fp(k_2400.curr).fp(k_2410.curr))
s.sweep(magnet.field, Bvec, delay=2.5)
print("HOLE RE-RUN DONE; field at", round(float(magnet.field()),3), "T")


PRECOND B=7.9027T  TG=-1.299 BG=-0.995 Si=0.000  T=1.267K


I=19.64 nA  at 7.9T:
  sr1: R=  1920.7Ohm   38%FS ok
  sr2: R=  1643.2Ohm   32%FS ok
  sr3: R= -1687.2Ohm   33%FS ok
Bvec[0]=7.900 -> Bvec[-1]=0.200  d(1/B)=0.01073 1/T


2026-06-21 06:34:36,372 [INFO] Starting sweep with ID 2


2026-06-21 06:34:36,373 [INFO] Minimum duration 0h 18m 57s


2026-06-21 07:58:17,284 [INFO] Completed in 1h 23m 40s


2026-06-21 07:58:17,286 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260621-QC0426\data\2\data.tsv.gz


HOLE RE-RUN DONE; field at 0.203 T


## HERA — electron spot-check setup: move to n=+3e12 cm⁻², D=0 [E1, Auto]

Stage the electron-side SdH spot check (investigation Step 3, other carrier sign). Move gates along D=0 to **n=+3.0e12 cm⁻²** → V_tg=+1.52, V_bg=+1.17 V (inside validators). Keep **coarse sens 1e-4** from the start (lesson from Run 1/2: size sens to the MR-inflated high-field Rxx, not low-field). Field at 0.2 T (from Run 2). Verify operating point + no overload before the ascending sweep.


In [24]:
import numpy as np, time
B = magnet.field()
print(f"PRECOND B={B:.4f}T T={heater.sample_temperature():.3f}K gates TG={dc205.voltage():.3f} BG={k_2400.volt():.3f} Si={k_2410.volt():.3f}")
assert abs(B-0.2) < 0.15, "field not near 0.2 T"
assert heater.sample_temperature() < 2.0
ratio=1.3063; c_tg=1.7027e-3; n_off=-2.391e15; e=1.602176634e-19
n_t=+3.0e16
Vbg=(n_t-n_off)*e/(2*ratio*c_tg); Vtg=ratio*Vbg
print(f"target n=+3e12 D=0 -> Vtg={Vtg:.3f} Vbg={Vbg:.3f}")
assert -6<Vtg<7 and -4.5<Vbg<4.5
k_2400.volt(Vbg); dc205_volt(Vtg); time.sleep(2)
sr1.sensitivity(1e-4); sr2.sensitivity(1e-4); sr3.sensitivity(1e-4); time.sleep(2)
Idr=sr0.X()/-1e6
print(f"I={Idr*1e9:.2f} nA at (n=+3e12, B=0.2T):")
for nm,sr in [("sr1",sr1),("sr2",sr2),("sr3",sr3)]:
    X=sr.X(); fs=abs(X)/sr.sensitivity()*100
    print(f"  {nm}: R={X/Idr:8.1f}Ohm  {fs:4.0f}%FS  {'OVLD' if fs>90 else 'ok'}")
print(f"leak I_bg={k_2400.curr()*1e12:.0f}pA I_si={k_2410.curr()*1e12:.0f}pA")


PRECOND B=0.2027T T=1.165K gates TG=-1.299 BG=-0.995 Si=0.000
target n=+3e12 D=0 -> Vtg=1.524 Vbg=1.167


I=19.99 nA at (n=+3e12, B=0.2T):
  sr1: R=    41.2Ohm     1%FS  ok
  sr2: R=    28.5Ohm     1%FS  ok
  sr3: R=    82.6Ohm     2%FS  ok


leak I_bg=242pA I_si=-21pA


## HERA — Run 3: electron SdH spot check n=+3e12 cm⁻², D=0 [E1, Auto]

Investigation Step 3, electron side. Uniform-1/B field sweep **ascending 0.2→7.9 T** (field at 0.2 T from Run 2), coarse sens 1e-4 (no clip; sized from MR-inflated high-field R). Same N=455, dwell 2.5 s. Read = 1/B FFT of Rxx vs F_L (=31 T at n=3e12), compared to the hole side and the ABA/ABC templates. Electron side is cleaner at low B (Rxx ~30–40 Ω). **Checkpoint ~40 min.** ~85 min; field ends at 7.9 T. Recipe: `sdh-spot-check`.


In [25]:
import numpy as np, time
B = magnet.field()
print(f"PRECOND B={B:.4f}T  TG={dc205.voltage():.3f} BG={k_2400.volt():.3f} Si={k_2410.volt():.3f} T={heater.sample_temperature():.3f}K")
assert abs(B-0.2) < 0.15, "field not at 0.2 T start"
assert abs(dc205.voltage()-1.524) < 0.02 and abs(k_2400.volt()-1.167) < 0.02, "gates not at n=+3e12 point"
assert abs(k_2410.volt()) < 0.01, "Si != 0"
assert sr1.sensitivity() == 1e-4 and sr3.sensitivity() == 1e-4, "sens not 1e-4"

for k,v in [('bias_voltage',sr1.amplitude()),('sr1_sens',sr1.sensitivity()),('sr2_sens',sr2.sensitivity()),
            ('sr3_sens',sr3.sensitivity()),('sample_temperature:',heater.sample_temperature()),('Si (V)',k_2410.volt())]:
    meas_config[k]=v
meas_config['comments']="SdH spot check ELECTRON n=+3e12 D=0; 2V/20nA sens1e-4; B 0.2->7.9T ascending uniform-1/B N=455 dwell2.5s; src T2 drn B7"

B_lo, B_hi, N = 0.2, 7.9, 455
Bvec = 1.0/np.linspace(1.0/B_lo, 1.0/B_hi, N)
print(f"Bvec[0]={Bvec[0]:.3f} -> Bvec[-1]={Bvec[-1]:.3f}")
assert Bvec.max() < 8.0

s = async_station(meas_config)
s.add_comment('sdh-spot-check')
s.add_comment('QC0426 ELECTRON n=+3e12 D=0; 2V/20nA sens1e-4; B 0.2->7.9T asc uniform-1/B N=455 dwell2.5s; src T2 drn B7')
(s.fp(sr0.X).fp(sr0.P).fp(sr1.X).fp(sr1.P).fp(sr2.X).fp(sr2.P).fp(sr3.X).fp(sr3.P).fp(k_2400.curr).fp(k_2410.curr))
s.sweep(magnet.field, Bvec, delay=2.5)
print("ELECTRON SPOT CHECK DONE; field at", round(float(magnet.field()),3), "T")


PRECOND B=0.2027T  TG=1.524 BG=1.167 Si=0.000 T=1.165K


Bvec[0]=0.200 -> Bvec[-1]=7.900
2026-06-21 08:09:17,657 [INFO] Starting sweep with ID 3


2026-06-21 08:09:17,658 [INFO] Minimum duration 0h 18m 57s


2026-06-21 09:32:59,389 [INFO] Completed in 1h 23m 41s


2026-06-21 09:32:59,390 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260621-QC0426\data\3\data.tsv.gz


ELECTRON SPOT CHECK DONE; field at 7.906 T


## HERA — Landau fan setup: worst-corner sensitivity check (n=−0.7e12, B=7.9 T) [E1, Auto]

Before the ~11 h fan, size the single fixed sensitivity at its **structural worst corner: low |n| × high B** (R∝1/n stacks with MR), where clipping would be unrecoverable (recipe). Magnet is at 7.9 T (Run 3 end), so measure it directly. Move gates to n=−0.7e12, D=0 (V_tg=−0.22, V_bg=−0.17), set sens 1e-3 (1 mV FS = 50 kΩ at 20 nA), read R + overload + leakage. If >80% FS, I'll coarsen to 2e-3 before the fan.


In [26]:
import numpy as np, time
from sharpelab_nb.gates import calc_gate_voltages
c_tg=1.7027e-3; c_bg=2.2243e-3; n_off=-0.2391  # n in 1e12 cm^-2 units, D_offset=0
Vtg0,Vbg0 = calc_gate_voltages(-0.7, 0.0, c_tg, c_bg, n_off, 0.0)
print(f"worst-corner gates n=-0.7e12,D=0: Vtg={float(Vtg0):.3f} Vbg={float(Vbg0):.3f}")
B=magnet.field(); print(f"B={B:.4f}T")
assert abs(B-7.9)<0.12, "magnet not at 7.9T"
assert -6<Vtg0<7 and -4.5<Vbg0<4.5
k_2400.volt(float(Vbg0)); dc205_volt(float(Vtg0)); time.sleep(2)
sr1.sensitivity(1e-3); sr2.sensitivity(1e-3); sr3.sensitivity(1e-3); time.sleep(3)
Idr=sr0.X()/-1e6
print(f"I={Idr*1e9:.2f}nA at WORST CORNER (n=-0.7e12, B=7.9T):")
for nm,sr in [("sr1",sr1),("sr2",sr2),("sr3",sr3)]:
    X=sr.X(); fs=abs(X)/sr.sensitivity()*100
    print(f"  {nm}: R={X/Idr:8.0f}ohm  {fs:5.1f}%FS  {'** OVLD **' if fs>90 else 'ok'}")
print(f"leak Ibg={k_2400.curr()*1e12:.0f}pA Isi={k_2410.curr()*1e12:.0f}pA")


worst-corner gates n=-0.7e12,D=0: Vtg=-0.217 Vbg=-0.166
B=7.9027T


I=19.38nA at WORST CORNER (n=-0.7e12, B=7.9T):
  sr1: R=   18212ohm   35.3%FS  ok
  sr2: R=   12819ohm   24.8%FS  ok
  sr3: R=   -9729ohm   18.9%FS  ok


leak Ibg=20pA Isi=-46pA


## HERA — Run 4: Landau fan (hole side, D=0) — the decisive F(n) test [E1, Auto, ~10–11 h]

Investigation Step 4/5 (`landau-fan-vti`). Per-column SdH-FFT fan: **B slow (descending 7.9→0.3 T), n-line fast (D=0)**. The measurement that settles ABA vs ABC — track each tone's F(n), test the Luttinger partition Σf=F_L(n) and the Onsager ML/BL curve shapes (separates real pockets from harmonics).

- **n-axis (fast):** n = −0.7 → −2.5e12 cm⁻², **100 cols**, D=0 via `calc_gate_voltages`. (n_max trimmed to 2.5e12 per the cost model; n=3e12 read from the Run-2 spot check, which a fan column at 2.5e12 cross-checks.)
- **B-axis (slow):** uniform-1/B 7.9→0.3 T, **N_B=250** = Nyquist Δ(1/B)≤1/(3·F_L(2.5e12)=25.9 T)=0.0129 /T over the 3.21 /T span. B_lo=0.3 T = measured hole onset (0.31).
- **Dynamic range:** single fixed **sens 2e-3** sized at the worst corner (low n × high B, measured 18 kΩ → 9% FS; headroom for QH ρxx peaks). High-n×low-B is grainy — acceptable (recipe); the spot check anchors absolute tones.
- **Settling:** fast_delay 1.2 s (4τ); slow_delay 1.0 s. Magnet blocking-ramp = the settle.
- **Cost:** 250×100 = 25k pts × ~1.4 s + ~1 h traverse ≈ **~10.5 h**. Field starts where the magnet sits (7.9 T) → descending, no re-traverse; ends at 0.3 T.
- **Checkpoints:** early (~25 min, first high-B rows = worst corner: clip/streak check), then ~every 3 h. Leakage (k_2400/k_2410 curr) in TSV; Lakeshore NOT polled (rig quirk).


In [ ]:
import numpy as np, time
from sharpelab_nb.gates import calc_gate_voltages
# --- Preconditions ---
B = magnet.field()
print(f"PRECOND B={B:.4f}T T={heater.sample_temperature():.3f}K Si={k_2410.volt():.3f}")
assert abs(B-7.9) < 0.12, "magnet not at 7.9 T start"
assert heater.sample_temperature() < 2.0
assert abs(k_2410.volt()) < 0.01, "Si != 0"

# --- Fast axis: D=0 hole n-line ---
c_tg=1.7027e-3; c_bg=2.2243e-3; n_off=-0.2391  # n in 1e12 cm^-2, D_offset=0
ns = np.linspace(-0.7, -2.5, 100)              # 1e12 cm^-2
Vtgs, Vbgs = calc_gate_voltages(ns, np.zeros_like(ns), c_tg, c_bg, n_off, 0.0)
Vtgs=np.asarray(Vtgs,float); Vbgs=np.asarray(Vbgs,float)
assert Vtgs.min()>-6 and Vtgs.max()<7 and Vbgs.min()>-4.5 and Vbgs.max()<4.5, "n-line out of gate range"
print(f"n-line: Vtg[{Vtgs.min():.3f},{Vtgs.max():.3f}] Vbg[{Vbgs.min():.3f},{Vbgs.max():.3f}]")

# --- Slow axis: B descending, uniform in 1/B (== field_invB(7.9,0.3,250)) ---
N_B=250
B_vec = 1.0/np.linspace(1.0/7.9, 1.0/0.3, N_B)
assert B_vec.max()<8.0 and abs(B_vec[0]-7.9)<0.02 and abs(B_vec[-1]-0.3)<0.02
FLmax=float(__import__('hera.analysis',fromlist=['luttinger_frequency']).luttinger_frequency(2.5e16))
dinvB=(1/0.3-1/7.9)/(N_B-1)
print(f"N_B={N_B} d(1/B)={dinvB:.5f} vs Nyquist 1/(3*F_L(2.5e12)={FLmax:.1f}T)={1/(3*FLmax):.5f} -> {'OK' if dinvB<=1/(3*FLmax) else 'UNDER'}")

# --- Sensitivity (worst-corner sized) ---
sr1.sensitivity(2e-3); sr2.sensitivity(2e-3); sr3.sensitivity(2e-3); time.sleep(2)

# --- meas_config ---
for k,v in [('bias_voltage',sr1.amplitude()),('sr1_sens',sr1.sensitivity()),('sr2_sens',sr2.sensitivity()),
            ('sr3_sens',sr3.sensitivity()),('sample_temperature:',heater.sample_temperature()),('Si (V)',k_2410.volt())]:
    meas_config[k]=v
meas_config['comments']="landau-fan hole D=0; n=-0.7..-2.5e12 (100); B 7.9->0.3T uniform-1/B N_B=250; 2V/20nA sens2e-3; src T2 drn B7"

# --- Build station + fan ---
s = async_station(meas_config)
s.add_comment('landau-fan-vti')
s.add_comment('QC0426 hole fan D=0; n=-0.7..-2.5e12 100col; B=7.9..0.3T 250row uniform-1/B; sens2e-3 2V/20nA; src T2 drn B7')
(s.fp(sr0.X).fp(sr0.P).fp(sr1.X).fp(sr1.P).fp(sr2.X).fp(sr2.P).fp(sr3.X).fp(sr3.P).fp(k_2400.curr).fp(k_2410.curr))
s.multimegasweep([magnet.field],[B_vec],[dc205_volt,k_2400.volt],[Vtgs,Vbgs], slow_delay=1.0, fast_delay=1.2)
dc205_volt(0.0); k_2400.volt(0.0)
print("FAN DONE; gates parked 0,0; field at", round(float(magnet.field()),3), "T")


## HERA — Run 4 fan (corrected: F_L computed inline; supersedes prior un-run cell)


In [ ]:
import numpy as np, time
from sharpelab_nb.gates import calc_gate_voltages
# --- Preconditions ---
B = magnet.field()
print(f"PRECOND B={B:.4f}T T={heater.sample_temperature():.3f}K Si={k_2410.volt():.3f}")
assert abs(B-7.9) < 0.12, "magnet not at 7.9 T start"
assert heater.sample_temperature() < 2.0
assert abs(k_2410.volt()) < 0.01, "Si != 0"

# --- Fast axis: D=0 hole n-line ---
c_tg=1.7027e-3; c_bg=2.2243e-3; n_off=-0.2391  # n in 1e12 cm^-2, D_offset=0
ns = np.linspace(-0.7, -2.5, 100)
Vtgs, Vbgs = calc_gate_voltages(ns, np.zeros_like(ns), c_tg, c_bg, n_off, 0.0)
Vtgs=np.asarray(Vtgs,float); Vbgs=np.asarray(Vbgs,float)
assert Vtgs.min()>-6 and Vtgs.max()<7 and Vbgs.min()>-4.5 and Vbgs.max()<4.5, "n-line out of gate range"
print(f"n-line: Vtg[{Vtgs.min():.3f},{Vtgs.max():.3f}] Vbg[{Vbgs.min():.3f},{Vbgs.max():.3f}]")

# --- Slow axis: B descending uniform in 1/B ---
N_B=250
B_vec = 1.0/np.linspace(1.0/7.9, 1.0/0.3, N_B)
assert B_vec.max()<8.0 and abs(B_vec[0]-7.9)<0.02 and abs(B_vec[-1]-0.3)<0.02
h=6.62607015e-34; e=1.602176634e-19
FLmax = h*2.5e16/(4*e)              # F_L at n_max=2.5e12
dinvB=(1/0.3-1/7.9)/(N_B-1)
print(f"N_B={N_B} d(1/B)={dinvB:.5f} vs Nyquist 1/(3*F_L({FLmax:.1f}T))={1/(3*FLmax):.5f} -> {'OK' if dinvB<=1/(3*FLmax) else 'UNDER'}")

# --- Sensitivity (worst-corner sized) ---
sr1.sensitivity(2e-3); sr2.sensitivity(2e-3); sr3.sensitivity(2e-3); time.sleep(2)

# --- meas_config ---
for k,v in [('bias_voltage',sr1.amplitude()),('sr1_sens',sr1.sensitivity()),('sr2_sens',sr2.sensitivity()),
            ('sr3_sens',sr3.sensitivity()),('sample_temperature:',heater.sample_temperature()),('Si (V)',k_2410.volt())]:
    meas_config[k]=v
meas_config['comments']="landau-fan hole D=0; n=-0.7..-2.5e12 (100); B 7.9->0.3T uniform-1/B N_B=250; 2V/20nA sens2e-3; src T2 drn B7"

# --- Build station + fan ---
s = async_station(meas_config)
s.add_comment('landau-fan-vti')
s.add_comment('QC0426 hole fan D=0; n=-0.7..-2.5e12 100col; B=7.9..0.3T 250row uniform-1/B; sens2e-3 2V/20nA; src T2 drn B7')
(s.fp(sr0.X).fp(sr0.P).fp(sr1.X).fp(sr1.P).fp(sr2.X).fp(sr2.P).fp(sr3.X).fp(sr3.P).fp(k_2400.curr).fp(k_2410.curr))
s.multimegasweep([magnet.field],[B_vec],[dc205_volt,k_2400.volt],[Vtgs,Vbgs], slow_delay=1.0, fast_delay=1.2)
dc205_volt(0.0); k_2400.volt(0.0)
print("FAN DONE; gates parked 0,0; field at", round(float(magnet.field()),3), "T")


PRECOND B=7.9027T T=1.272K Si=0.000
n-line: Vtg[-1.064,-0.217] Vbg[-0.814,-0.166]
N_B=250 d(1/B)=0.01288 vs Nyquist 1/(3*F_L(25.8T))=0.01290 -> OK


2026-06-21 10:00:59,929 [INFO] Starting multimegasweep with ID 4


2026-06-21 10:00:59,930 [INFO] Minimum duration 8h 24m 10s
